# Batched `kkt_solve_many` performance (pounce#77)

After a `Solver.solve` converges, the LDLᵀ factor of the final KKT
system is cached on the solver. `kkt_solve(rhs)` does one back-solve
through that factor; `kkt_solve_many(rhs_flat, nrhs)` does `nrhs`
back-solves through the same factor in one call.

Until pounce#77 the `_many` entry point looped single-RHS solves
internally, so it bought nothing over the Python loop. pounce#77
rewires it as a 3-tier dispatch:

1. **Tier 1 — inline flat-slice path** (`solve_many_cached_flat`).
   Slack / `z` / `v` blocks downcast to `DenseVector` and the
   bound-expansion matrices to `ExpansionMatrix` once at the top;
   Phase 1 (aug-RHS build with scatter-add-div) and Phase 3 (bound-
   multiplier expansion) then run as raw loops on flat slices with
   no `dyn` dispatch in the per-RHS inner loops.
2. **Tier 2 — closure-based path** (`solve_many_cached`) when the
   13-tag cache check on `w, j_c, j_d, z_l/u, v_l/u, slack_x/s_l/u,
   sigma_x/s` misses the flat-slice fast path.
3. **Tier 3 — per-RHS fallback** when the cache misses entirely.

Phase 2 (the inner Aug-system solve) in all three tiers dispatches
through a single `TSymLinearSolver::multi_solve` with `nrhs > 1`, so
the FERAL backend can amortise per-call setup and run BLAS-3 panel
kernels (feral#57) and batched iterative refinement (feral#58) on
the back-substitution.

This notebook drives two converged QPs to a stable factor and times
looped `kkt_solve` vs batched `kkt_solve_many` at a sweep of `nrhs`.
A `ratio` below 1 (or `speedup` above 1) means batched wins.

In [1]:
import time
import numpy as np
from pounce import Problem, Solver

## 1. Sparse QP — diagonal Hessian + dense Jacobian rows

$$
\min_x\;\tfrac12\|x-p\|^2 \quad \text{s.t.}\quad A x = b,\;\; -10^6\le x\le 10^6
$$

with $A\in\mathbb R^{4\times n}$ dense. The KKT factor is sparse —
supernodes are small — so this is the *control* case: the win comes
almost entirely from amortising Phase 1/3 overhead, not BLAS-3.

We solve once to converge the IPM, then beat on the cached factor.

In [2]:
def build_sparse_qp(n: int, n_eq: int = 4) -> Solver:
    rng = np.random.default_rng(0)
    A = rng.standard_normal((n_eq, n))
    b = rng.standard_normal((n_eq,))
    p = rng.standard_normal((n,))

    class QP:
        def objective(self, x):    return 0.5 * float(np.sum((x - p) ** 2))
        def gradient(self, x):     return x - p
        def constraints(self, x):  return A @ x - b
        def jacobianstructure(self):
            return (np.repeat(np.arange(n_eq), n),
                    np.tile(np.arange(n), n_eq))
        def jacobian(self, x):     return A.reshape(-1)
        def hessianstructure(self): return np.arange(n), np.arange(n)
        def hessian(self, x, lam, of): return of * np.ones(n)

    prob = Problem(
        n=n, m=n_eq, problem_obj=QP(),
        lb=np.full(n, -1e6), ub=np.full(n, 1e6),
        cl=np.zeros(n_eq), cu=np.zeros(n_eq),
    )
    prob.add_option("tol", 1e-10)
    prob.add_option("print_level", 0)
    prob.add_option("sb", "yes")
    solver = Solver(prob)
    solver.solve(x0=np.zeros(n))
    return solver


def time_block(fn, repeats: int = 7) -> float:
    fn()  # warm
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return min(ts)


def sweep(solver: Solver, label: str, nrhs_list=(1, 4, 16, 64, 256)):
    kkt_dim = solver.kkt_dim
    rng = np.random.default_rng(1)
    rhs_one = rng.standard_normal(kkt_dim)
    t_one = time_block(lambda: solver.kkt_solve(rhs_one))
    print(f"{label}  (kkt_dim = {kkt_dim})")
    print(f"  {'N':>4}  {'one µs':>10}  {'many µs':>11}  {'ratio':>7}  {'speedup':>8}")
    for N in nrhs_list:
        rhs_many = rng.standard_normal(N * kkt_dim)
        t_many = time_block(lambda Nl=N, r=rhs_many: solver.kkt_solve_many(r, Nl))
        ratio = t_many / (N * t_one)
        speedup = 1.0 / ratio
        print(f"  {N:4d}  {t_one*1e6:10.2f}  {t_many*1e6:11.2f}  {ratio:7.3f}  {speedup:8.2f}×")

In [3]:
for n in [512, 2048]:
    solver = build_sparse_qp(n)
    sweep(solver, f"Sparse QP  n={n}")

Sparse QP  n=512  (kkt_dim = 1540)
     N      one µs      many µs    ratio   speedup
     1       67.08        59.42    0.886      1.13×
     4       67.08       236.67    0.882      1.13×
    16       67.08       599.79    0.559      1.79×
    64       67.08      2295.96    0.535      1.87×
   256       67.08      9273.67    0.540      1.85×
Sparse QP  n=2048  (kkt_dim = 6148)
     N      one µs      many µs    ratio   speedup
     1      219.62       211.71    0.964      1.04×
     4      219.62       819.87    0.933      1.07×
    16      219.62      2394.00    0.681      1.47×


    64      219.62      9574.08    0.681      1.47×


   256      219.62     37454.17    0.666      1.50×


At `N=1` the ratio sits near 1 (one batched call ≈ one single call —
no amortising headroom). As `N` grows, the inline flat-slice path
amortises Phase 1/3 setup and the single `multi_solve` lets FERAL
drop the per-call overhead it would otherwise pay `N` times. The
sparse case caps out at the per-RHS speedup the workspace +
Phase 1/3 amortisation can buy on a small-supernode factor.

## 2. Dense-Hessian QP — exercises feral BLAS-3 panel kernels

The sparse QP's factor has supernodes too small for feral's BLAS-3
panel kernels to amortise (feral#57's `fwd_blas3` / `back_blas3`).
Swap in a dense Hessian $H = M^\top M + I$ and the augmented KKT
factor gets one large dense supernode of size ~n; forward and back
substitution each become a single TRSM on an $n\times \text{nrhs}$
panel — exactly what those kernels were written for.

This is also the case where the feral#58 fix matters: with
iterative refinement enabled (the FERAL default), the pre-#58
`solve_many_refined` looped per column and threw away the panel
kernel; with #58 the refinement loop runs *batched* residual +
batched panel solve.

In [4]:
def build_dense_h_qp(n: int, n_eq: int = 4) -> Solver:
    rng = np.random.default_rng(0)
    A = rng.standard_normal((n_eq, n))
    b = rng.standard_normal((n_eq,))
    p = rng.standard_normal((n,))
    M = rng.standard_normal((n, n)) / np.sqrt(n)
    H = M.T @ M + np.eye(n)
    iu, ju = np.triu_indices(n)
    H_upper = H[iu, ju]

    class QP:
        def objective(self, x):    return 0.5 * float(x @ H @ x) - float(p @ x)
        def gradient(self, x):     return H @ x - p
        def constraints(self, x):  return A @ x - b
        def jacobianstructure(self):
            return (np.repeat(np.arange(n_eq), n),
                    np.tile(np.arange(n), n_eq))
        def jacobian(self, x):     return A.reshape(-1)
        def hessianstructure(self): return iu, ju
        def hessian(self, x, lam, of): return of * H_upper

    prob = Problem(
        n=n, m=n_eq, problem_obj=QP(),
        lb=np.full(n, -1e6), ub=np.full(n, 1e6),
        cl=np.zeros(n_eq), cu=np.zeros(n_eq),
    )
    prob.add_option("tol", 1e-10)
    prob.add_option("print_level", 0)
    prob.add_option("sb", "yes")
    solver = Solver(prob)
    solver.solve(x0=np.zeros(n))
    return solver


solver = build_dense_h_qp(512)
sweep(solver, "Dense-H QP  n=512", nrhs_list=(1, 4, 16, 64, 256))

Dense-H QP  n=512  (kkt_dim = 1540)
     N      one µs      many µs    ratio   speedup
     1      353.00       341.71    0.968      1.03×
     4      353.00      1601.21    1.134      0.88×
    16      353.00      4297.08    0.761      1.31×


    64      353.00     16273.58    0.720      1.39×


   256      353.00     66234.50    0.733      1.36×

## 3. What you should see

Per-RHS time for `kkt_solve_many` drops below `kkt_solve` looped at
wide `nrhs`. Numbers depend on machine and BLAS, but the qualitative
shape is stable:

* `N=1` ratio ≈ 1.0 (no amortising room — both paths do one Aug-
  system solve and one Phase 1/3 pass).
* Ratio drops monotonically with `N` up to `N≈64` then plateaus.
* Sparse QP plateau is dominated by Phase 1/3 / dispatch amortising
  (Tier 1's inline-slice path).
* Dense-H QP plateau drops further because feral's BLAS-3 panel
  kernels actually fire on the large supernode.

If you see `ratio ≥ 1` at `N≥16` on dense-H, the FERAL build is
missing one of the two upstream fixes (feral#57 panel kernels,
feral#58 batched refinement). Toggle `POUNCE_FERAL_REFINE=0` in the
environment to isolate #58.

## When to reach for it

* **JAX-side backward passes** through `JaxProblem` (`jacrev` of a
  parametric solve): the cotangent loop in the implicit-diff VJP
  goes through `kkt_solve_many` with one column per output, so a
  Jacobian with many outputs (e.g. `n_out > 16`) amortises the
  factor reuse across columns.
* **Sensitivity batches**: any caller that already has `nrhs > 1`
  RHSes against a cached KKT factor — pass them all at once and
  drop the Python loop.
* **Not the right tool** when `nrhs == 1` (`kkt_solve` is fine), the
  factor isn't cached (`solve_many_cached_flat` falls to Tier 3 and
  the win disappears), or when you actually want a single residual
  evaluation per RHS — `kkt_solve_many` shares the factor, not the
  preceding KKT-matrix build.